# 법안 분야 분류 파인튜닝 (C조 입법 영향 지도 「빼곡」)

미리 많은 한국어 글로 학습된 언어모델(`klue/roberta-small`)을 가져와 법안 분야 분류에 맞게 조금 더 학습(파인튜닝, 미세조정)하고, [분야 분류 노트북](field_classifier.ipynb)의 규칙 기반·로지스틱 회귀·Dense 신경망과 **같은 분할 10번**에서 정확도를 비교한다.
화면에 쓰는 모델은 바꾸지 않고 비교 기록만 남긴다 (9절).

**Colab GPU에서 실행 (런타임 → 런타임 유형 변경 → T4 GPU). GPU가 없으면 CPU로도 돌지만 오래 걸린다.**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cbnu-lawmap/law-impact-map/blob/main/notebooks/field_finetune.ipynb)

**순서**: 0 왜 파인튜닝인가 → 1 문제 → 2 데이터 → 3 사전학습 모델 불러오기 → 4 한 번 학습해 보기 → 5 분할 10번 비교 → 6 혼동행렬·오답 → 7 결과표 → 8 한계 → 9 화면과의 관계

## 0. 왜 파인튜닝인가 (14주차)

분야 분류 노트북은 글을 **TF-IDF**(글자 2~3개 묶음이 얼마나 자주 나오는지)로 숫자 3,000개로 바꾸고, 라벨 298건만으로 처음부터 학습했다. 이 방식은 낱말의 순서와 문맥을 버린다.

**파인튜닝(미세조정)** 은 다르게 한다.

1. 많은 한국어 글로 미리 학습해 둔 언어모델(**사전학습 모델**)을 가져온다. 이 모델은 이미 문장을 읽고 뜻을 숫자로 바꾸는 법을 안다.
2. 맨 위에 분야 9개짜리 출력층(Softmax 9)만 새로 붙인다.
3. 우리 라벨로 모델 전체를 **작은 학습률로 몇 epoch 만** 더 학습한다. 이미 배운 것을 크게 망가뜨리지 않고 우리 문제에 맞춘다.

| | 분야 분류 노트북 | 이 노트북 |
|---|---|---|
| 글 → 숫자 | TF-IDF 3,000차원 (규칙으로 셈) | 사전학습 언어모델 (미리 학습됨) |
| 학습하는 것 | Softmax 층(로지스틱 회귀) 또는 Dense 층 | 언어모델 전체 + 새 Softmax 9 층 |
| 처음 값 | 무작위 | 사전학습 가중치 (출력층만 무작위) |
| 학습 장치 | CPU 몇 초 | GPU 권장 |

**질문**: 라벨이 300건뿐일 때, 사전학습 모델을 파인튜닝하면 TF-IDF + 로지스틱 회귀보다 나은가? 테스트 60건 한 번으로는 1~2건 차이로 숫자가 흔들리므로, 분야 분류 노트북 8절과 **같은 분할 10번**에서 평균과 표준편차로 답한다.

## 1. 문제

**무엇을**: 법안 1건의 제안이유 글을 넣으면 9개 분야 중 어느 분야인지 고른다 (분야 분류 노트북과 같은 문제, 같은 라벨).

- 입력: 제안이유 글 (머리말 "제안이유 및 주요내용"을 뗀 본문)
- 출력: 9개 분야 각각의 확률 (합 = 1). 가장 큰 것을 분야로 쓴다.

| 분야 번호 | 분야 |
|---|---|
| 1 | 소방·화재 |
| 2 | 재난·안전관리 |
| 3 | 건축물 안전·노후·유지관리 |
| 4 | 주택 공급·임대·금융 |
| 5 | 공동주택 관리·주거환경 |
| 6 | 정비사업·도시재생 |
| 7 | 도시계획·건축규제 |
| 8 | 주차·보행·교통 |
| 9 | 건설산업·기술·기타 |

In [ ]:
# 준비: Colab 에는 torch·transformers 가 이미 있어 다시 설치하지 않는다 (없을 때만 설치)
import os, sys, importlib.util
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'          # 사용 통계 보내지 않음
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'     # GPU 행렬 계산을 되도록 같은 순서로 (torch 를 불러오기 전에 정해야 한다)
if importlib.util.find_spec('torch') is None:
    !pip install -q torch --index-url https://download.pytorch.org/whl/cpu
if importlib.util.find_spec('transformers') is None:
    !pip -q install transformers==4.46.3
if 'google.colab' in sys.modules and not os.path.exists('/usr/share/fonts/truetype/nanum'):
    !apt-get -qq install fonts-nanum > /dev/null

In [ ]:
import re, json, time, copy, gc, random, datetime, warnings, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix
from IPython.display import Markdown, display
warnings.filterwarnings('ignore')
transformers.logging.set_verbosity_error()           # "출력층을 새로 만들었다" 같은 안내 숨기기

import matplotlib.font_manager as fm
_f = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
if os.path.exists(_f):
    fm.fontManager.addfont(_f)                       # 세션 다시 시작 없이 한글 글꼴 등록
    plt.rcParams['font.family'] = fm.FontProperties(fname=_f).get_name()
plt.rcParams['axes.unicode_minus'] = False

# ---- 설정 (4절에서 시간이 모자라면 EPOCHS·MAX_LEN 만 줄인다. 분할 수는 줄이지 않는다) ----
MAX_LEN = 256          # 법안 1건에서 읽는 토큰(낱말 조각) 수. 넘으면 뒤를 자른다
EPOCHS = 5             # 학습 데이터를 몇 번 돌지
LR = 3e-5              # 학습률 (사전학습 가중치를 크게 망가뜨리지 않게 작게)
BATCH = 16
VAL_FRAC = 0.1         # 훈련 데이터 중 검증용(가장 좋은 epoch 고르기)으로 떼는 비율
WARMUP_FRAC = 0.1      # 처음 10% 걸음 동안 학습률을 0에서 LR 까지 올리고, 그 뒤 0까지 곧게 내린다
SEED = 42
N_SPLIT = 10           # 분야 분류 노트북 8절과 같은 분할 10번 (random_state = 100 + r)
TIME_BUDGET_MIN = 30   # 분할 10번 전체 예상 시간이 이보다 길면 EPOCHS·MAX_LEN 을 줄인다
MODEL_CANDIDATES = ['klue/roberta-small', 'klue/bert-base']   # 앞의 것을 못 받으면 다음 것

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU'
FIELD = {1:'소방·화재', 2:'재난·안전관리', 3:'건축물 안전·노후·유지관리', 4:'주택 공급·임대·금융',
         5:'공동주택 관리·주거환경', 6:'정비사업·도시재생', 7:'도시계획·건축규제', 8:'주차·보행·교통', 9:'건설산업·기술·기타'}
LABEL2ID = {k: i for i, k in enumerate(FIELD)}       # 분야 1~9 → 출력 자리 0~8 (분야 분류 노트북과 같은 순서)
ID2LABEL = {i: k for k, i in LABEL2ID.items()}
print('torch', torch.__version__, '/ transformers', transformers.__version__)
print('장치:', DEVICE, '/', GPU_NAME)
if DEVICE.type == 'cpu':
    print('GPU 가 없다. CPU 로도 돌지만 오래 걸린다 (런타임 → 런타임 유형 변경 → T4 GPU 권장)')

## 2. 데이터

분야 분류 노트북과 **같은 파일, 같은 읽기 방법, 같은 전처리**를 쓴다 (`load()`, `clean()` 을 그대로 옮김).

- `labels_300.csv`: 사람이 분야를 붙인 법안 300건 (LABEL 1~9, 학습 제외 표시 EXCLUDE, 애매 표시 AMBIG)

GitHub 저장소 main 의 `data/` 에서 읽는다. 인터넷이 안 되면 `NB_LOCAL_DATA` 폴더, 그다음 `../data/` 를 읽고, 그래도 없으면 (Colab) 파일 올리기 창을 띄운다.

In [ ]:
BASE = 'https://raw.githubusercontent.com/cbnu-lawmap/law-impact-map/main/data/'
LOCAL_DIRS = [d for d in [os.environ.get('NB_LOCAL_DATA'), '../data', '.'] if d]

def load(name, required=True):
    try:
        d = pd.read_csv(BASE + name, encoding='utf-8-sig', dtype={'BILL_NO': str})
        print(name, '← GitHub main')
        return d
    except Exception:
        pass
    for folder in LOCAL_DIRS:
        p = os.path.join(folder, name)
        if os.path.exists(p):
            print(name, '←', p)
            return pd.read_csv(p, encoding='utf-8-sig', dtype={'BILL_NO': str})
    if not required:
        return None
    from google.colab import files
    print(name, '파일을 올려 주세요'); files.upload()
    return pd.read_csv(name, encoding='utf-8-sig', dtype={'BILL_NO': str})

lab = load('labels_300.csv')        # 라벨 300건
print('라벨', lab.shape)

In [ ]:
def clean(t):
    t = str(t)
    t = re.sub(r'^\s*제안이유\s*(및\s*주요내용)?', '', t)
    t = re.sub(r'\s+', ' ', t)
    return t.strip()

def law_name(n):
    n = re.sub(r'\s*(일부|전부)개정법률안.*|\s*법률안.*', '', str(n))
    return re.sub(r'\(.*?\)', '', n).strip()

df = lab[lab['EXCLUDE'] == 'N'].copy()
df['TEXT'] = df['SUMMARY'].map(clean)
df['LAW'] = df['BILL_NAME'].map(law_name)
df = df.reset_index(drop=True)
df['Y'] = df['LABEL'].map(LABEL2ID)                  # 정답 자리 0~8
print('학습에 쓰는 법안:', len(df), '건 (제외', (lab['EXCLUDE'] != 'N').sum(), '건)')
print('분야별 수:', df['LABEL'].value_counts().sort_index().to_dict())

### 2-1. 기준선 설정을 분야 분류 노트북과 같게

로지스틱 회귀는 분야 분류 노트북과 같은 TF-IDF(글자 2~3개 묶음, 3,000개)와 같은 규제 세기 C 를 쓴다. C 는 그 노트북 5절과 똑같이 첫 분할(random_state = 42) 의 훈련 데이터에서 5겹 교차검증으로 다시 고르고, 내보낸 파일(`web/data/models/field_classifier.json`)의 값과 같은지 본다. Dense 신경망 숫자도 그 파일에서 읽는다 (이 노트북에서는 TensorFlow 를 쓰지 않는다).

In [ ]:
def make_vec():
    return TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 3), max_features=3000, sublinear_tf=True)

def rule_fit(d):
    return d.groupby('LAW')['LABEL'].agg(lambda s: s.value_counts().idxmax()).to_dict(), d['LABEL'].value_counts().idxmax()

def rule_predict(r, d):
    table, default = r
    return d['LAW'].map(table).fillna(default).astype(int).values

tr0, _ = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df['LABEL'])
_v = make_vec()
_X = _v.fit_transform(tr0['TEXT']).toarray().astype('float32')
gs = GridSearchCV(LogisticRegression(max_iter=5000), {'C': [0.1, 1, 10, 100]},
                  cv=StratifiedKFold(5, shuffle=True, random_state=SEED))
gs.fit(_X, tr0['LABEL'])
LR_C = gs.best_params_['C']

FC_URL = 'https://raw.githubusercontent.com/cbnu-lawmap/law-impact-map/main/web/data/models/field_classifier.json'
FC_LOCAL = ['../web/data/models/field_classifier.json', 'field_classifier.json']
def load_fc():
    try:
        with urllib.request.urlopen(FC_URL, timeout=30) as r:
            print('field_classifier.json ← GitHub main')
            return json.loads(r.read().decode('utf-8'))
    except Exception:
        pass
    for p in FC_LOCAL:
        if os.path.exists(p):
            print('field_classifier.json ←', p)
            return json.load(open(p, encoding='utf-8'))
    print('field_classifier.json 을 읽지 못함 (Dense 신경망 줄은 비워 둔다)')
    return None

FC = load_fc()
FC_M = (FC or {}).get('metrics', {})
print('고른 C =', LR_C, '/ field_classifier.json 의 C =', FC_M.get('logreg_C'))
if FC_M.get('logreg_C') is not None and float(FC_M['logreg_C']) != float(LR_C):
    print('주의: C 가 내보낸 파일과 다르다. 이 노트북은 여기서 고른 C 를 쓴다.')
print('Dense 신경망 (분야 분류 노트북 8절, 분할 10번):', FC_M.get('cv_mean_nn'), '±', FC_M.get('cv_std_nn'), '/ 은닉층', FC_M.get('nn_hidden'))

## 3. 사전학습 모델 불러오기

`klue/roberta-small`: KLUE(한국어 자연어 이해 평가) 팀이 공개한 한국어 RoBERTa 의 작은 판. 받지 못하면 `klue/bert-base` 를 쓴다.

- **토크나이저**: 글을 낱말 조각(토큰)으로 자르고 번호로 바꾼다. 이미지의 픽셀처럼 모델에 넣는 숫자다.
- **`AutoModelForSequenceClassification`**: 언어모델 위에 분야 9개짜리 출력층을 붙인 모델. 출력층만 무작위로 시작하고 나머지는 사전학습 가중치다.

In [ ]:
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def new_model(name, seed):
    set_seed(seed)                                   # 새 출력층의 처음 값도 시드로 고정
    m = AutoModelForSequenceClassification.from_pretrained(
        name, num_labels=9, id2label={i: str(k) for i, k in ID2LABEL.items()}, label2id={str(k): i for k, i in LABEL2ID.items()})
    return m.to(DEVICE)

MODEL_NAME, tokenizer = None, None
for cand in MODEL_CANDIDATES:
    try:
        tokenizer = AutoTokenizer.from_pretrained(cand)
        _m = new_model(cand, SEED)
        MODEL_NAME = cand
        break
    except Exception as e:
        print(cand, '불러오기 실패:', type(e).__name__, str(e)[:200])
assert MODEL_NAME is not None, '사전학습 모델을 받지 못했다 (인터넷 연결 확인)'
N_PARAMS = sum(p.numel() for p in _m.parameters())
N_HEAD = sum(p.numel() for n, p in _m.named_parameters() if 'classifier' in n)
print('모델:', MODEL_NAME, '/ 파라미터 %s개 (그중 새 출력층 %s개)' % (format(N_PARAMS, ','), format(N_HEAD, ',')))
del _m; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

ex = df['TEXT'].iloc[0]
print('글 앞부분:', ex[:60])
print('토큰 앞부분:', tokenizer.tokenize(ex)[:15])
n_tok = df['TEXT'].map(lambda t: len(tokenizer(t, add_special_tokens=True)['input_ids']))
print('법안별 토큰 수: 중앙값 %d / 최대 %d / MAX_LEN %d 를 넘어 뒤가 잘리는 법안 %.0f%%'
      % (n_tok.median(), n_tok.max(), MAX_LEN, 100 * (n_tok > MAX_LEN).mean()))

### 3-1. 학습 함수

설명하기 쉽게 Hugging Face `Trainer` 대신 PyTorch 학습 반복을 직접 쓴다.

1. 글을 토큰 번호로 바꾼다 (`MAX_LEN` 까지, 짧으면 빈칸 채움).
2. 최적화: **AdamW** (Adam + 가중치 감쇠), 학습률은 처음 10% 걸음 동안 0 → `LR` 로 올리고(워밍업) 끝까지 0 으로 곧게 내린다.
3. 손실: 교차 엔트로피 (9주차와 같음).
4. epoch 마다 검증 데이터로 정확도·손실을 재고, **검증 정확도가 가장 좋은 epoch** 의 가중치를 기억해 둔다 (같으면 검증 손실이 낮은 쪽). 테스트 정확도는 그 가중치로 잰다. 분야 분류 노트북 7절의 조기 종료(가장 좋았던 가중치로 돌아가기)와 같은 생각이다.

In [ ]:
_enc_cache = {}
def encode(max_len):
    """df 전체를 한 번에 토큰 번호로 바꿔 둔다 (MAX_LEN 이 바뀌면 다시)."""
    if max_len not in _enc_cache:
        e = tokenizer(list(df['TEXT']), max_length=max_len, truncation=True, padding='max_length', return_tensors='pt')
        _enc_cache[max_len] = (e['input_ids'], e['attention_mask'])
    return _enc_cache[max_len]

def batches(idx, max_len, shuffle, seed=0):
    ids, mask = encode(max_len)
    idx = np.array(idx)
    if shuffle:
        idx = np.random.RandomState(seed).permutation(idx)
    y = torch.tensor(df['Y'].values)
    for i in range(0, len(idx), BATCH):
        j = torch.tensor(idx[i:i + BATCH])
        yield ids[j].to(DEVICE), mask[j].to(DEVICE), y[j].to(DEVICE)

@torch.no_grad()
def evaluate(model, idx, max_len):
    model.eval()
    losses, preds, n = 0.0, [], 0
    for ids, mask, y in batches(idx, max_len, shuffle=False):
        logits = model(input_ids=ids, attention_mask=mask).logits
        losses += torch.nn.functional.cross_entropy(logits, y, reduction='sum').item()
        preds.append(logits.argmax(1).cpu().numpy()); n += len(y)
    p = np.concatenate(preds)
    return losses / n, accuracy_score(df['Y'].values[np.array(idx)], p), p

def linear_warmup(opt, n_warm, n_total):
    def f(step):
        if step < n_warm:
            return (step + 1) / max(1, n_warm)
        return max(0.0, (n_total - step) / max(1, n_total - n_warm))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)

def split_r(r):
    """분야 분류 노트북 8절과 같은 분할: 테스트 20%, random_state = 100 + r, 분야 비율 유지."""
    a, b = train_test_split(df, test_size=0.2, random_state=100 + r, stratify=df['LABEL'])
    return a, b

def finetune(r, epochs, max_len, verbose=True):
    a, b = split_r(r)
    try:
        fit, val = train_test_split(a, test_size=VAL_FRAC, random_state=r, stratify=a['LABEL'])
    except ValueError:                               # 분야 하나가 너무 적어 비율을 못 맞추면 그냥 나눈다
        fit, val = train_test_split(a, test_size=VAL_FRAC, random_state=r)
    fit_i, val_i, te_i = fit.index.values, val.index.values, b.index.values
    model = new_model(MODEL_NAME, SEED + r)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = epochs * int(np.ceil(len(fit_i) / BATCH))
    sched = linear_warmup(opt, int(WARMUP_FRAC * steps), steps)
    hist, best, best_key = [], None, None
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train()
        tl, tn = 0.0, 0
        for ids, mask, y in batches(fit_i, max_len, shuffle=True, seed=SEED + 1000 * r + ep):
            loss = torch.nn.functional.cross_entropy(model(input_ids=ids, attention_mask=mask).logits, y)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()
            tl += loss.item() * len(y); tn += len(y)
        vl, va, _ = evaluate(model, val_i, max_len)
        hist.append({'epoch': ep, 'train_loss': tl / tn, 'val_loss': vl, 'val_acc': va})
        if verbose:
            print('  분할 %d epoch %d  학습 손실 %.3f / 검증 손실 %.3f / 검증 정확도 %.3f' % (r, ep, tl / tn, vl, va))
        key = (va, -vl)
        if best_key is None or key > best_key:      # 검증 정확도가 높을수록, 같으면 검증 손실이 낮을수록
            best_key, best_ep = key, ep
            best = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    train_sec = time.time() - t0
    model.load_state_dict(best)
    _, te_acc, te_pred = evaluate(model, te_i, max_len)
    del model, opt, best; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return {'split': r, 'random_state': 100 + r, 'ft_acc': te_acc, 'best_epoch': best_ep, 'train_seconds': train_sec,
            'total_seconds': time.time() - t0, 'hist': pd.DataFrame(hist), 'test_idx': te_i, 'test_pred': te_pred,
            'n_fit': len(fit_i), 'n_val': len(val_i), 'n_test': len(te_i)}

## 4. 한 번 학습해 보기

첫 분할(분할 0, random_state = 100)로 한 번 학습하고 epoch 마다 학습 손실과 검증 손실을 본다. 이 한 번에 걸린 시간으로 분할 10번 전체 시간을 미리 어림한다.

**시간 맞추기**: 10배 한 예상 시간이 30분을 넘으면 (GPU 가 없을 때 등) `EPOCHS` 와 `MAX_LEN` 을 줄인다. 이때 분할 0 도 줄인 설정으로 **다시 학습**해서 10번이 모두 같은 설정이 되게 한다. 분할 수(10번)는 줄이지 않는다. 실제로 쓴 값은 결과 파일에 적는다.

In [ ]:
EPOCHS_USED, MAX_LEN_USED = EPOCHS, MAX_LEN
REDUCED, REDUCE_REASON = False, ''
print('분할 0 학습 (EPOCHS %d, MAX_LEN %d)' % (EPOCHS_USED, MAX_LEN_USED))
res0 = finetune(0, EPOCHS_USED, MAX_LEN_USED)
t_split = res0['total_seconds']
projected = t_split * N_SPLIT
print('분할 0: %.0f초 → 분할 %d번 예상 %.1f분 (한도 %d분)' % (t_split, N_SPLIT, projected / 60, TIME_BUDGET_MIN))

if projected > TIME_BUDGET_MIN * 60:
    # 걸리는 시간은 대략 epoch 수 × 토큰 수에 비례한다고 보고, 한도 안에 드는 첫 설정을 고른다
    options = [(e, L) for e, L in [(4, 256), (3, 256), (3, 192), (3, 128), (2, 128), (2, 96), (2, 64), (1, 64)]
               if e <= EPOCHS and L <= MAX_LEN]
    est = lambda e, L: projected * (e * L) / (EPOCHS * MAX_LEN)
    fit_opt = [o for o in options if est(*o) <= TIME_BUDGET_MIN * 60]
    EPOCHS_USED, MAX_LEN_USED = fit_opt[0] if fit_opt else options[-1]
    REDUCED = True
    REDUCE_REASON = ('%s 에서 분할 0 이 %.0f초 걸려 10번 예상 %.1f분 > %d분. EPOCHS %d→%d, MAX_LEN %d→%d 로 줄이고 분할 0 부터 다시 학습 (줄인 설정 예상 %.1f분)'
                     % (GPU_NAME, t_split, projected / 60, TIME_BUDGET_MIN, EPOCHS, EPOCHS_USED, MAX_LEN, MAX_LEN_USED,
                        est(EPOCHS_USED, MAX_LEN_USED) / 60))
    print(REDUCE_REASON)
    res0 = finetune(0, EPOCHS_USED, MAX_LEN_USED)
print('쓰는 설정: EPOCHS %d, MAX_LEN %d%s' % (EPOCHS_USED, MAX_LEN_USED, ' (줄임)' if REDUCED else ''))
print('분할 0 테스트 정확도 %.3f (가장 좋은 검증 epoch %d) / 학습 %d건, 검증 %d건, 테스트 %d건'
      % (res0['ft_acc'], res0['best_epoch'], res0['n_fit'], res0['n_val'], res0['n_test']))

In [ ]:
h = res0['hist']
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(h['epoch'], h['train_loss'], marker='o', label='학습')
ax[0].plot(h['epoch'], h['val_loss'], marker='o', label='검증')
ax[0].axvline(res0['best_epoch'], color='gray', ls=':', label='고른 epoch')
ax[0].set_title('손실'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].plot(h['epoch'], h['val_acc'], marker='o', color='C1')
ax[1].axvline(res0['best_epoch'], color='gray', ls=':')
ax[1].set_ylim(0, 1); ax[1].set_title('검증 정확도 (%d건)' % res0['n_val']); ax[1].set_xlabel('epoch')
ax[0].set_xticks(h['epoch']); ax[1].set_xticks(h['epoch'])
plt.suptitle('%s 파인튜닝, 분할 0 (EPOCHS %d, MAX_LEN %d)' % (MODEL_NAME, EPOCHS_USED, MAX_LEN_USED)); plt.tight_layout(); plt.show()

rise = h['val_loss'].iloc[-1] > h['val_loss'].min() + 1e-6
display(Markdown(
    f"**실행 결과**: 학습 손실은 {h['train_loss'].iloc[0]:.3f} → {h['train_loss'].iloc[-1]:.3f}, "
    f"검증 손실은 {h['val_loss'].iloc[0]:.3f} → {h['val_loss'].iloc[-1]:.3f} (가장 낮을 때 {h['val_loss'].min():.3f}, epoch {int(h['val_loss'].idxmin()) + 1}). "
    + ("검증 손실이 가장 낮은 뒤 다시 올라 마지막 epoch 쪽에서는 과적합 기미가 있다. " if rise else "검증 손실이 끝까지 내려가 이 epoch 수에서는 과적합이 뚜렷하지 않다. ")
    + f"검증 정확도가 가장 좋은 epoch {res0['best_epoch']}의 가중치로 테스트 {res0['n_test']}건을 재면 {res0['ft_acc']:.3f}이다. "
    f"검증 데이터가 {res0['n_val']}건뿐이라 1건이 {1 / res0['n_val']:.3f}만큼 정확도를 바꾼다."))

## 5. 분할 10번 비교

분야 분류 노트북 8절과 **같은 분할 10번**(테스트 20%, random_state = 100 + r, 분야 비율 유지)에서

- **파인튜닝**: 훈련 80% 중 10%(분야 비율 유지, random_state = r)를 검증용으로 떼고 나머지로 학습, 가장 좋은 검증 epoch 의 가중치로 테스트.
- **로지스틱 회귀**: 그 노트북과 같게 훈련 80% 전체로 TF-IDF 와 모델을 맞추고 테스트 (같은 C).
- **규칙 기반**: 그 노트북과 같게 훈련 80% 에서 법률명별로 가장 많은 분야 → 테스트.

분할 0 은 4절 결과를 그대로 쓴다 (같은 설정).

In [ ]:
results = {0: res0}
rows, lr_pred, rule_pred = [], {}, {}
lr_seconds, rule_seconds, LR_N_COEF, RULE_N = 0.0, 0.0, None, []
t_all = time.time()
for r in range(N_SPLIT):
    a, b = split_r(r)
    t0 = time.time()
    v = make_vec()
    xa = v.fit_transform(a['TEXT']).toarray().astype('float32'); xb = v.transform(b['TEXT']).toarray().astype('float32')
    lr_r = LogisticRegression(C=LR_C, max_iter=5000).fit(xa, a['LABEL'])
    lr_pred[r] = lr_r.predict(xb)
    lr_seconds += time.time() - t0
    LR_N_COEF = int(lr_r.coef_.size + lr_r.intercept_.size); LR_SHAPE = lr_r.coef_.shape
    t0 = time.time()
    rf = rule_fit(a)
    rule_pred[r] = rule_predict(rf, b)
    rule_seconds += time.time() - t0
    RULE_N.append(len(rf[0]))
    if r not in results:
        results[r] = finetune(r, EPOCHS_USED, MAX_LEN_USED, verbose=False)
    res = results[r]
    assert (res['test_idx'] == b.index.values).all()  # 세 방법이 같은 테스트 법안을 본다
    y = b['LABEL'].values
    rows.append({'split': r, 'random_state': 100 + r,
                 'ft_acc': float(res['ft_acc']), 'lr_acc': float(accuracy_score(y, lr_pred[r])),
                 'rule_acc': float(accuracy_score(y, rule_pred[r])),
                 'best_epoch': int(res['best_epoch']), 'train_seconds': round(float(res['train_seconds']), 1)})
    print('분할 %d  파인튜닝 %.3f / 로지스틱 회귀 %.3f / 규칙 %.3f  (가장 좋은 epoch %d, 학습 %.0f초)'
          % (r, rows[-1]['ft_acc'], rows[-1]['lr_acc'], rows[-1]['rule_acc'], res['best_epoch'], res['train_seconds']))
per = pd.DataFrame(rows)
print('분할 10번 전체 %.1f분' % ((time.time() - t_all + res0['total_seconds']) / 60))
per

In [ ]:
diff = per['ft_acc'] - per['lr_acc']          # 같은 분할끼리 뺀 차이 (짝지은 차이)
ft_mean, ft_std = per['ft_acc'].mean(), per['ft_acc'].std()
lr_mean, lr_std = per['lr_acc'].mean(), per['lr_acc'].std()
rule_mean, rule_std = per['rule_acc'].mean(), per['rule_acc'].std()
d_mean, d_std = diff.mean(), diff.std()
TOTAL_TRAIN_SEC = float(per['train_seconds'].sum())

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(per['split'], per['ft_acc'], marker='o', label='파인튜닝')
ax[0].plot(per['split'], per['lr_acc'], marker='s', label='로지스틱 회귀')
ax[0].plot(per['split'], per['rule_acc'], marker='^', color='#999', label='규칙 기반')
ax[0].set_xticks(per['split']); ax[0].set_xlabel('분할'); ax[0].set_ylabel('테스트 정확도'); ax[0].set_ylim(0.4, 1)
ax[0].set_title('분할별 테스트 정확도 (60건)'); ax[0].legend(fontsize=8)
ax[1].bar(per['split'], diff, color=['#4a7ab5' if d >= 0 else '#c0504d' for d in diff])
ax[1].axhline(0, color='k', lw=0.8); ax[1].axhline(d_mean, color='gray', ls='--', label='평균 %.3f' % d_mean)
ax[1].set_xticks(per['split']); ax[1].set_xlabel('분할'); ax[1].set_title('파인튜닝 − 로지스틱 회귀'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()
print('파인튜닝 %.3f ± %.3f / 로지스틱 회귀 %.3f ± %.3f / 짝지은 차이 %.3f ± %.3f (파인튜닝이 이긴 분할 %d번, 같음 %d번)'
      % (ft_mean, ft_std, lr_mean, lr_std, d_mean, d_std, (diff > 0).sum(), (diff == 0).sum()))

## 6. 혼동행렬·오답 몇 건

분할 10번의 테스트 예측을 **모두 모아**(10 × 60 = 600개, 법안 하나가 여러 분할의 테스트에 들어갈 수 있음) 파인튜닝 모델의 혼동행렬을 그린다. 한 분할 60건보다 칸마다 수가 많아 어느 분야끼리 헷갈리는지 잘 보인다.

In [ ]:
pool = pd.concat([pd.DataFrame({'split': r, 'idx': results[r]['test_idx'],
                                'PRED': [ID2LABEL[int(p)] for p in results[r]['test_pred']],
                                'LR': lr_pred[r]}) for r in range(N_SPLIT)], ignore_index=True)
pool = pool.join(df[['BILL_NO', 'BILL_NAME', 'LABEL', 'TEXT', 'AMBIG']], on='idx')
cm = confusion_matrix(pool['LABEL'], pool['PRED'], labels=list(FIELD))
fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(9)); ax.set_yticks(range(9))
ax.set_xticklabels(list(FIELD), fontsize=9); ax.set_yticklabels([f'{k} {v}' for k, v in FIELD.items()], fontsize=8)
for i in range(9):
    for j in range(9):
        if cm[i, j]: ax.text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=8)
ax.set_xlabel('예측'); ax.set_ylabel('정답'); ax.set_title('혼동행렬 (%s 파인튜닝, 분할 10번 테스트 %d개 모음)' % (MODEL_NAME, len(pool)))
plt.tight_layout(); plt.show()

off = cm.copy(); np.fill_diagonal(off, 0)
i, j = np.unravel_index(off.argmax(), off.shape)
per_field = pd.Series(cm.diagonal() / cm.sum(1), index=[FIELD[k] for k in FIELD])
display(Markdown(
    f"**실행 결과**: 모은 예측 {len(pool)}개의 정확도는 {(pool['PRED'] == pool['LABEL']).mean():.3f}이다. "
    f"가장 많이 헷갈린 쌍은 정답 '{FIELD[i + 1]}'을 '{FIELD[j + 1]}'로 고른 {off[i, j]}개다. "
    f"분야별 정확도가 가장 낮은 분야는 '{per_field.idxmin()}'({per_field.min():.2f}), 가장 높은 분야는 '{per_field.idxmax()}'({per_field.max():.2f})이다."))

In [ ]:
# 틀린 법안 3건 (서로 다른 법안, 분할 순서대로 앞에서부터)
wrong = pool[pool['PRED'] != pool['LABEL']].drop_duplicates('BILL_NO').head(3).copy()
wrong['정답'] = wrong['LABEL'].map(FIELD); wrong['파인튜닝 예측'] = wrong['PRED'].map(FIELD); wrong['로지스틱 회귀'] = wrong['LR'].map(FIELD)
wrong['제안이유(앞부분)'] = wrong['TEXT'].str[:120]
WRONG_EXAMPLES = [{'split': int(w.split), 'bill_no': w.BILL_NO, 'bill_name': w.BILL_NAME, 'true': int(w.LABEL), 'true_name': FIELD[int(w.LABEL)],
                   'pred': int(w.PRED), 'pred_name': FIELD[int(w.PRED)], 'logreg_pred': int(w.LR), 'ambig': w.AMBIG, 'summary_head': w.TEXT[:120]}
                  for w in wrong.itertuples()]
print('파인튜닝 오답 (모은 예측 %d개 중 %d개, 서로 다른 법안 %d건)' % (len(pool), (pool['PRED'] != pool['LABEL']).sum(),
      pool.loc[pool['PRED'] != pool['LABEL'], 'BILL_NO'].nunique()))
pd.set_option('display.max_colwidth', 130)
wrong[['split', 'BILL_NAME', '정답', '파인튜닝 예측', '로지스틱 회귀', 'AMBIG', '제안이유(앞부분)']]

## 7. 결과표

모두 같은 분할 10번의 테스트 정확도 평균 ± 표준편차다. 규칙 기반과 로지스틱 회귀는 이 노트북에서 다시 쟀고, Dense 신경망은 분야 분류 노트북 8절의 숫자(`field_classifier.json` 의 `cv_mean_nn`, `cv_std_nn`)를 읽어 왔다. 표와 해설은 이 실행의 숫자로 코드가 만든다.

In [ ]:
def fmt_sec(s):
    return '%.1f초' % s if s < 60 else '%.1f분' % (s / 60)

nn_mean, nn_std = FC_M.get('cv_mean_nn'), FC_M.get('cv_std_nn')
nn_hidden = FC_M.get('nn_hidden')
if nn_hidden is not None and FC and 'input' in FC:
    _sizes = [len(FC['input']['idf'])] + list(nn_hidden) + [9]
    nn_params = format(sum(i * o + o for i, o in zip(_sizes[:-1], _sizes[1:])), ',') + ' (Dense %s → 9, json 의 nn_hidden 으로 계산)' % '-'.join(map(str, nn_hidden))
else:
    nn_params = 'field_classifier.ipynb 참고'
nn_time = fmt_sec(FC_M['nn_train_seconds']) if 'nn_train_seconds' in FC_M else 'field_classifier.ipynb 참고'

table = pd.DataFrame([
    {'방법': '규칙 기반 (법률명 → 가장 많은 분야)', '10회 평균 정확도 ± 표준편차': '%.3f ± %.3f' % (rule_mean, rule_std),
     '학습 시간': '%s (10번 합계)' % fmt_sec(rule_seconds), '모델 크기(파라미터 수)': '규칙 %d~%d개 (법률명 표)' % (min(RULE_N), max(RULE_N))},
    {'방법': '로지스틱 회귀 (TF-IDF, C=%g)' % LR_C, '10회 평균 정확도 ± 표준편차': '%.3f ± %.3f' % (lr_mean, lr_std),
     '학습 시간': '%s (10번 합계, CPU, TF-IDF 포함)' % fmt_sec(lr_seconds), '모델 크기(파라미터 수)': '%s (계수 %s×%d + 절편 %d)' % (format(LR_N_COEF, ','), format(LR_SHAPE[1], ','), LR_SHAPE[0], LR_SHAPE[0])},
    {'방법': 'Dense 신경망 (분야 분류 노트북)', '10회 평균 정확도 ± 표준편차': ('%.3f ± %.3f' % (nn_mean, nn_std)) if nn_mean is not None else '읽지 못함',
     '학습 시간': nn_time, '모델 크기(파라미터 수)': nn_params},
    {'방법': '파인튜닝 (%s)' % MODEL_NAME, '10회 평균 정확도 ± 표준편차': '%.3f ± %.3f' % (ft_mean, ft_std),
     '학습 시간': '%s (10번 합계, %s)' % (fmt_sec(TOTAL_TRAIN_SEC), GPU_NAME), '모델 크기(파라미터 수)': format(N_PARAMS, ',')},
])
table

In [ ]:
# 해설: 손으로 적지 않고 위 숫자로 만든다
higher = ft_mean > lr_mean
bigger = abs(d_mean) > d_std
lines = [
    '| 방법 | 10회 평균 정확도 ± 표준편차 | 학습 시간 | 모델 크기(파라미터 수) |',
    '|---|---|---|---|',
] + ['| %s | %s | %s | %s |' % tuple(row) for row in table.values] + [
    '',
    '설정: %s, EPOCHS %d, MAX_LEN %d, 학습률 %g, 배치 %d, 검증 비율 %.0f%%, 시드 %d%s. 장치: %s.'
    % (MODEL_NAME, EPOCHS_USED, MAX_LEN_USED, LR, BATCH, 100 * VAL_FRAC, SEED, (' (시간 때문에 줄임: ' + REDUCE_REASON + ')') if REDUCED else '', GPU_NAME),
    '',
    '**읽는 법**',
    '',
    ('- 파인튜닝의 10회 평균(%.3f)이 로지스틱 회귀(%.3f)보다 **높다**.' if higher else
     '- 파인튜닝의 10회 평균(%.3f)이 로지스틱 회귀(%.3f)보다 **낮다**. 사전학습 모델을 써도 이 데이터에서는 TF-IDF + 로지스틱 회귀를 넘지 못했다.' if ft_mean < lr_mean else
     '- 파인튜닝의 10회 평균(%.3f)과 로지스틱 회귀(%.3f)가 같다.') % (ft_mean, lr_mean),
    '- 같은 분할끼리 뺀 차이(파인튜닝 − 로지스틱 회귀)의 평균은 %+.3f, 그 표준편차는 %.3f 이다. **차이가 표준편차보다 %s.** %s'
    % (d_mean, d_std, '크다' if bigger else '작다',
       '분할을 바꿔도 같은 쪽이 대체로 이긴다는 뜻이다.' if bigger else '분할에 따라 이기는 쪽이 바뀔 수 있어 우열이 뚜렷하지 않다.'),
    '- 파인튜닝이 이긴 분할 %d번, 진 분할 %d번, 같은 분할 %d번.' % ((diff > 0).sum(), (diff < 0).sum(), (diff == 0).sum()),
    '- 두 방법 각각의 표준편차는 파인튜닝 %.3f, 로지스틱 회귀 %.3f 이다. 같은 테스트 법안을 쓰므로 둘의 표준편차보다 짝지은 차이의 표준편차가 비교에 더 알맞다.' % (ft_std, lr_std),
    '- 파인튜닝은 학습 시간이 로지스틱 회귀의 약 %.0f배, 파라미터 수가 약 %.0f배다.' % (TOTAL_TRAIN_SEC / max(lr_seconds, 1e-9), N_PARAMS / LR_N_COEF),
]
if nn_mean is not None:
    lines.append('- Dense 신경망(%.3f ± %.3f)보다 파인튜닝이 %s.' % (nn_mean, nn_std, '높다' if ft_mean > nn_mean else '낮다' if ft_mean < nn_mean else '같다'))
lines.append('- 가장 좋은 검증 epoch: %s (분할 0~9).' % ', '.join(map(str, per['best_epoch'])))
display(Markdown('\n'.join(lines)))

## 8. 한계

In [ ]:
display(Markdown('\n'.join([
    '- **라벨이 적다**: 298건, 분할마다 학습 %d건·검증 %d건·테스트 %d건이다. 테스트 1건이 정확도 %.3f, 검증 1건이 %.3f 를 바꾼다.'
    % (res0['n_fit'], res0['n_val'], res0['n_test'], 1 / res0['n_test'], 1 / res0['n_val']),
    '- **조건이 완전히 같지는 않다**: 파인튜닝은 가장 좋은 epoch 를 고르려고 훈련 데이터의 %.0f%% 를 검증용으로 떼어 학습에 덜 쓴다. 로지스틱 회귀·Dense 신경망은 훈련 80%% 전체로 학습한다 (분야 분류 노트북과 같게).' % (100 * VAL_FRAC),
    '- **설정을 찾지 않았다**: 학습률·epoch·MAX_LEN 은 흔히 쓰는 값 하나만 썼다%s. 로지스틱 회귀는 C 를 교차검증으로 골랐으니 파인튜닝 쪽이 덜 다듬어졌다.'
    % ((' (그나마 시간 때문에 EPOCHS %d, MAX_LEN %d 로 줄임)' % (EPOCHS_USED, MAX_LEN_USED)) if REDUCED else ''),
    '- **글을 자른다**: MAX_LEN %d 토큰을 넘는 법안은 뒷부분을 읽지 못한다 (3절: 토큰 수 중앙값 %d).' % (MAX_LEN_USED, n_tok.median()),
    '- **GPU 에서는 시드를 고정해도 셋째 자리가 조금 달라질 수 있다**: 같은 GPU 종류에서도 계산 순서 때문에 작은 차이가 날 수 있다. 결과표는 실행한 숫자로 코드가 만든다.',
    '- **Dense 신경망 숫자는 다른 실행에서 왔다**: 같은 분할 10번이지만 분야 분류 노트북이 따로 잰 값(`field_classifier.json`)이다.',
    '- **화면에 바로 쓸 수 없다**: 서버(`server/lib/nn.js`)는 TF-IDF + Dense 계산만 하고, 파라미터 %s개 트랜스포머를 돌리려면 다른 서버 구성이 필요하다.' % format(N_PARAMS, ','),
])))

## 9. 화면과의 관계

화면에 쓰는 모델은 바꾸지 않는다. 더 나으면 다음 단계에서 서버 교체 검토, 지금은 비교 기록만.

지금 화면(지도·행정 화면 분야 필터, 에이전트 `classify_field`)은 분야 분류 노트북이 고른 모델(`field_classifier.json` 의 `metrics.model`)을 그대로 쓴다. 이 노트북은 모델 가중치를 저장소에 저장하지 않고, 비교 숫자만 `data/finetune_compare.json` 에 남긴다.

In [ ]:
if ft_mean - lr_mean > d_std:
    msg = ('파인튜닝(%s)이 로지스틱 회귀보다 10회 평균 %.3f 높고, 이 차이가 짝지은 차이의 표준편차(%.3f)보다 크다. '
           '다음 단계에서 서버 교체(트랜스포머를 돌릴 서버 구성, 응답 시간, 비용)를 검토할 근거가 된다.' % (MODEL_NAME, ft_mean - lr_mean, d_std))
elif ft_mean > lr_mean:
    msg = ('파인튜닝이 10회 평균으로는 %.3f 높지만 짝지은 차이의 표준편차(%.3f)보다 작아, 서버를 바꿀 만큼 뚜렷한 차이는 아니다.' % (ft_mean - lr_mean, d_std))
else:
    msg = ('파인튜닝이 로지스틱 회귀보다 10회 평균 %.3f 낮다. 화면에는 지금 모델을 그대로 쓴다.' % (lr_mean - ft_mean))
display(Markdown('**이 실행의 결론**: ' + msg))

In [ ]:
# 비교 결과 저장 (모델 가중치는 저장하지 않는다)
out = {
    'model': MODEL_NAME,
    'config': {'max_len': MAX_LEN_USED, 'epochs': EPOCHS_USED, 'lr': LR, 'batch': BATCH, 'val_frac': VAL_FRAC, 'seed': SEED,
               'warmup_frac': WARMUP_FRAC, 'optimizer': 'AdamW(weight_decay=0.01)', 'schedule': 'linear warmup + linear decay',
               'max_len_planned': MAX_LEN, 'epochs_planned': EPOCHS, 'reduced': REDUCED, 'reduce_reason': REDUCE_REASON,
               'time_budget_min': TIME_BUDGET_MIN, 'best_epoch_rule': 'val_acc 최대, 같으면 val_loss 최소',
               'splits': 'train_test_split(test_size=0.2, random_state=100+r, stratify=LABEL), r=0..9 (field_classifier 8절과 같음)',
               'logreg_C': float(LR_C)},
    'per_split': [{k: (float(v) if isinstance(v, (float, np.floating)) else int(v)) for k, v in row.items()
                   if k in ('split', 'random_state', 'ft_acc', 'lr_acc', 'rule_acc', 'best_epoch', 'train_seconds')}
                  for row in per.to_dict('records')],
    'ft_mean': round(float(ft_mean), 4), 'ft_std': round(float(ft_std), 4),
    'lr_mean': round(float(lr_mean), 4), 'lr_std': round(float(lr_std), 4),
    'rule_mean': round(float(rule_mean), 4), 'rule_std': round(float(rule_std), 4),
    'nn_mean': nn_mean, 'nn_std': nn_std, 'nn_source': 'web/data/models/field_classifier.json metrics.cv_mean_nn / cv_std_nn',
    'paired_diff_mean': round(float(d_mean), 4), 'paired_diff_std': round(float(d_std), 4),
    'diff_exceeds_std': bool(abs(d_mean) > d_std),
    'total_train_seconds': round(TOTAL_TRAIN_SEC, 1), 'logreg_train_seconds': round(lr_seconds, 2),
    'params': int(N_PARAMS), 'logreg_params': LR_N_COEF,
    'device': DEVICE.type, 'gpu_name': GPU_NAME,
    'torch_version': torch.__version__, 'transformers_version': transformers.__version__,
    'run_date': datetime.date.today().isoformat(),
    'n_labels': int(len(df)),
    'wrong_examples': WRONG_EXAMPLES,
}
data_dir = '../data' if os.path.isdir('../data') else '.'
OUT_PATH = os.path.join(data_dir, 'finetune_compare.json')
with open(OUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(out, f, ensure_ascii=False, indent=1)
print('저장:', OUT_PATH, '/ 파인튜닝 %.3f ± %.3f, 로지스틱 회귀 %.3f ± %.3f, 차이 %+.3f ± %.3f'
      % (out['ft_mean'], out['ft_std'], out['lr_mean'], out['lr_std'], out['paired_diff_mean'], out['paired_diff_std']))

In [ ]:
# Colab 이면 결과 파일을 내려받는다 (저장소에 커밋하려고)
if 'google.colab' in sys.modules:
    try:
        from google.colab import files
        files.download(OUT_PATH)
    except Exception as e:
        print('내려받기 실패:', e, '/ 왼쪽 파일 창에서', OUT_PATH, '를 직접 받으세요')